# 08 - Trajectory Simulations and Parameter Sweeps

This notebook performs comprehensive SGD trajectory simulations with parameter sweeps to understand how hyperparameters affect optimization dynamics.

**Converted from:** `trajectories_simulation.nb` (Mathematica)

## Contents:
1. Multiple trajectory simulations
2. Learning rate sweep analysis
3. Batch size impact study
4. Statistical analysis of convergence
5. Variance and bias analysis

## Background

Understanding how SGD hyperparameters affect optimization is crucial. This notebook systematically explores the parameter space to reveal key relationships between learning rate, batch size, and convergence behavior.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
from tqdm import tqdm

# Add utils to path
sys.path.insert(0, str(Path.cwd() / 'utils'))

from sgd_simulator import SGDSimulator, SGDConfig
from loss_functions import SmoothNonlinearLoss, generate_noisy_data
from visualization import plot_loss_landscape, plot_trajectories, plot_parameter_evolution

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print("✓ Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")

## 1. Setup: Data and Loss Function

We'll use the same smooth nonlinear loss function from previous notebooks.

In [ ]:
# Generate data
np.random.seed(42)

x_data, y_data = generate_noisy_data(
    x_range=(-3, 3),
    n_points=20,
    n_samples_per_point=5,
    noise_std=0.5,
    p=1.0,
    random_state=42
)

loss_obj = SmoothNonlinearLoss(p=1.0)

print(f"Generated {len(x_data)} data points")
print(f"Loss function ready")

## 2. Multiple Trajectory Simulations

Run multiple SGD trajectories from different random initializations to study variability.

In [ ]:
# Configuration
base_config = SGDConfig(
    learning_rate=0.05,
    batch_size=10,
    n_iterations=3000,
    random_state=42
)

# Run multiple trajectories
n_runs = 10
trajectories = []
final_losses = []
final_params = []

np.random.seed(123)
simulator = SGDSimulator(base_config)

for i in range(n_runs):
    # Random initialization
    init_params = np.random.uniform(-0.5, 2.5, size=2)
    
    # Run trajectory
    traj, iters = simulator.run_trajectory(
        initial_params=init_params,
        gradient_fn=loss_obj.gradient,
        x_data=x_data,
        y_data=y_data,
        save_every=20
    )
    
    trajectories.append(traj)
    final_params.append(traj[-1])
    final_losses.append(loss_obj(traj[-1], x_data, y_data))

final_params = np.array(final_params)
final_losses = np.array(final_losses)

print(f"Completed {n_runs} trajectory simulations")
print(f"Mean final loss: {np.mean(final_losses):.4f} ± {np.std(final_losses):.4f}")
print(f"Best final loss: {np.min(final_losses):.4f}")
print(f"Worst final loss: {np.max(final_losses):.4f}")

In [ ]:
# Visualize all trajectories
fig, ax = plt.subplots(figsize=(14, 11))

# Loss landscape
param_range = ((-1, 3), (-1, 3))
plot_loss_landscape(
    loss_fn=loss_obj,
    x_data=x_data,
    y_data=y_data,
    param_range=param_range,
    n_points=100,
    contour_levels=30,
    ax=ax
)

# Plot all trajectories
colors = plt.cm.tab10(np.linspace(0, 1, n_runs))
for i, traj in enumerate(trajectories):
    ax.plot(traj[:, 0], traj[:, 1], '-', linewidth=2, 
           color=colors[i], alpha=0.7)
    ax.plot(traj[0, 0], traj[0, 1], 'o', markersize=8, 
           color=colors[i], markeredgecolor='black', markeredgewidth=1.5)
    ax.plot(traj[-1, 0], traj[-1, 1], 's', markersize=8,
           color=colors[i], markeredgecolor='black', markeredgewidth=1.5)

ax.set_title(f'Multiple SGD Trajectories (n={n_runs})', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Analyze final parameter distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Parameter scatter
axes[0].scatter(final_params[:, 0], final_params[:, 1], 
               c=final_losses, cmap='viridis', s=200, 
               edgecolor='black', linewidth=2)
axes[0].set_xlabel('Parameter a', fontsize=12)
axes[0].set_ylabel('Parameter b', fontsize=12)
axes[0].set_title('Final Parameter Distribution', fontsize=14)
axes[0].grid(True, alpha=0.3)
cbar = plt.colorbar(axes[0].collections[0], ax=axes[0])
cbar.set_label('Final Loss', fontsize=11)

# Loss histogram
axes[1].hist(final_losses, bins=8, edgecolor='black', alpha=0.7)
axes[1].axvline(np.mean(final_losses), color='red', linestyle='--', 
               linewidth=2, label='Mean')
axes[1].set_xlabel('Final Loss', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of Final Losses', fontsize=14)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Learning Rate Sweep

Systematic study of how learning rate affects convergence speed and final loss.

In [ ]:
# Learning rate sweep
learning_rates = [0.001, 0.005, 0.01, 0.03, 0.05, 0.1, 0.15]
lr_results = {}

initial_params = np.array([0.5, 2.0])  # Fixed initialization

for lr in learning_rates:
    config = SGDConfig(
        learning_rate=lr,
        batch_size=10,
        n_iterations=3000,
        random_state=42
    )
    
    simulator = SGDSimulator(config)
    traj, iters = simulator.run_trajectory(
        initial_params=initial_params,
        gradient_fn=loss_obj.gradient,
        x_data=x_data,
        y_data=y_data,
        save_every=10
    )
    
    # Compute loss trajectory
    losses = np.array([loss_obj(params, x_data, y_data) for params in traj])
    
    lr_results[lr] = {
        'trajectory': traj,
        'iterations': iters,
        'losses': losses,
        'final_loss': losses[-1]
    }

print("Learning rate sweep completed")
for lr, result in lr_results.items():
    print(f"LR={lr:.3f}: Final loss={result['final_loss']:.4f}")

In [ ]:
# Plot loss evolution for different learning rates
fig, ax = plt.subplots(figsize=(14, 7))

colors_lr = plt.cm.plasma(np.linspace(0.1, 0.9, len(learning_rates)))
for i, (lr, result) in enumerate(lr_results.items()):
    ax.plot(result['iterations'], result['losses'], 
           linewidth=2.5, color=colors_lr[i], alpha=0.8, label=f'LR={lr}')

ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Loss Evolution for Different Learning Rates', fontsize=14)
ax.set_yscale('log')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot trajectories for different learning rates
fig, ax = plt.subplots(figsize=(14, 11))

# Loss landscape
plot_loss_landscape(
    loss_fn=loss_obj,
    x_data=x_data,
    y_data=y_data,
    param_range=param_range,
    n_points=100,
    contour_levels=30,
    ax=ax
)

# Plot trajectories
for i, (lr, result) in enumerate(lr_results.items()):
    traj = result['trajectory']
    ax.plot(traj[:, 0], traj[:, 1], '-', linewidth=2.5, 
           color=colors_lr[i], alpha=0.8, label=f'LR={lr}')
    ax.plot(traj[0, 0], traj[0, 1], 'o', markersize=10, 
           color=colors_lr[i], markeredgecolor='black', markeredgewidth=2)

ax.set_title('Trajectories for Different Learning Rates', fontsize=14)
ax.legend(fontsize=11, loc='upper right')
plt.tight_layout()
plt.show()

print("Observe: Higher LR leads to faster but noisier convergence")

## 4. Batch Size Impact Study

Investigate how batch size affects gradient noise and convergence.

In [ ]:
# Batch size sweep
batch_sizes = [5, 10, 20, 50, len(x_data)]  # Full batch at end
bs_results = {}

fixed_lr = 0.05

for bs in batch_sizes:
    config = SGDConfig(
        learning_rate=fixed_lr,
        batch_size=bs,
        n_iterations=3000,
        random_state=42
    )
    
    simulator = SGDSimulator(config)
    traj, iters = simulator.run_trajectory(
        initial_params=initial_params,
        gradient_fn=loss_obj.gradient,
        x_data=x_data,
        y_data=y_data,
        save_every=10
    )
    
    losses = np.array([loss_obj(params, x_data, y_data) for params in traj])
    
    bs_results[bs] = {
        'trajectory': traj,
        'iterations': iters,
        'losses': losses,
        'final_loss': losses[-1]
    }

print("Batch size sweep completed")
for bs, result in bs_results.items():
    print(f"BS={bs}: Final loss={result['final_loss']:.4f}")

In [ ]:
# Plot loss evolution for different batch sizes
fig, ax = plt.subplots(figsize=(14, 7))

colors_bs = plt.cm.viridis(np.linspace(0.1, 0.9, len(batch_sizes)))
for i, (bs, result) in enumerate(bs_results.items()):
    label = f'BS={bs}' if bs < len(x_data) else 'Full Batch'
    ax.plot(result['iterations'], result['losses'], 
           linewidth=2.5, color=colors_bs[i], alpha=0.8, label=label)

ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title('Loss Evolution for Different Batch Sizes', fontsize=14)
ax.set_yscale('log')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Smaller batches show more noise but can escape local minima")

## 5. Statistical Analysis of Convergence

Analyze variance and statistical properties across multiple runs.

In [ ]:
# Run multiple trials with same configuration
n_trials = 20
all_loss_curves = []

config = SGDConfig(
    learning_rate=0.05,
    batch_size=10,
    n_iterations=3000,
    random_state=None  # Different random seed each time
)

for trial in range(n_trials):
    simulator = SGDSimulator(config)
    init_params = np.random.uniform(-0.5, 2.5, size=2)
    
    traj, iters = simulator.run_trajectory(
        initial_params=init_params,
        gradient_fn=loss_obj.gradient,
        x_data=x_data,
        y_data=y_data,
        save_every=10
    )
    
    losses = np.array([loss_obj(params, x_data, y_data) for params in traj])
    all_loss_curves.append(losses)

# Align all curves to same length
min_len = min(len(curve) for curve in all_loss_curves)
all_loss_curves = np.array([curve[:min_len] for curve in all_loss_curves])

# Compute statistics
mean_loss = np.mean(all_loss_curves, axis=0)
std_loss = np.std(all_loss_curves, axis=0)
median_loss = np.median(all_loss_curves, axis=0)

print(f"Analyzed {n_trials} independent trials")

In [ ]:
# Plot statistical summary
fig, ax = plt.subplots(figsize=(14, 7))

iterations_common = iters[:min_len]

# Plot individual curves (faint)
for curve in all_loss_curves:
    ax.plot(iterations_common, curve, 'gray', alpha=0.2, linewidth=1)

# Plot mean and confidence interval
ax.plot(iterations_common, mean_loss, 'b-', linewidth=3, label='Mean', alpha=0.9)
ax.fill_between(iterations_common, 
                mean_loss - std_loss, mean_loss + std_loss,
                color='blue', alpha=0.2, label='±1 std')
ax.plot(iterations_common, median_loss, 'r--', linewidth=2.5, 
       label='Median', alpha=0.8)

ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Loss', fontsize=12)
ax.set_title(f'Statistical Analysis of SGD Convergence ({n_trials} trials)', fontsize=14)
ax.set_yscale('log')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final loss: {mean_loss[-1]:.4f} ± {std_loss[-1]:.4f}")

## Summary

In this notebook, we performed:

1. **Multiple Trajectory Simulations**: Explored variability across different initializations
2. **Learning Rate Sweep**: Found that higher learning rates converge faster but with more noise
3. **Batch Size Impact**: Smaller batches add noise that aids exploration
4. **Statistical Analysis**: Quantified mean, variance, and confidence intervals across runs

**Key Findings:**
- Learning rate controls the speed-noise tradeoff in convergence
- Smaller batch sizes introduce gradient noise that helps escape poor local minima
- There is significant run-to-run variability depending on initialization
- Mean convergence curves mask substantial variance in individual trajectories

**Next Steps:**
- Explore enhanced trajectory simulations with improved visualization
- Study multi-dimensional examples in 2D and 3D